# 🚀 DataMate — Zalo AI Challenge 2026: Public Test Inference

Notebook hoàn chỉnh giúp bạn chạy toàn bộ pipeline suy luận trên tập **`public_test`** (1,000 câu hỏi) bằng **GPU Google Colab** (A100 / L4 / T4) và xuất file **`submission.jsonl`** chuẩn để nộp lên bảng xếp hạng.

> 💡 **Lưu ý:** Trước khi chạy, hãy chọn GPU: **Runtime** → **Change runtime type** → chọn **A100 GPU** (hoặc L4 / T4).

In [ ]:
# 1.1 Kiểm tra thông tin GPU đang kết nối
!nvidia-smi

In [ ]:
# 1.2 Cài đặt các thư viện phụ thuộc cần thiết
!pip install -q duckdb>=1.1.0 pandas>=2.2.0 openpyxl python-calamine
!pip install -q transformers>=4.48.0 accelerate>=1.0.0 bitsandbytes>=0.43.0
!pip install -q tqdm loguru unidecode regex

---  
### 2. Kết nối Google Drive & Đồng bộ Code GitHub

In [ ]:
# 2.1 Mount Google Drive và trỏ vào thư mục làm việc
from google.colab import drive
drive.mount('/content/drive')

# Chuyển vào thư mục chứa dữ liệu trên Google Drive
%cd /content/drive/MyDrive/ZALO_AI_Challenge

In [ ]:
# 🔄 2.2 ĐỒNG BỘ CODE MỚI NHẤT TỪ GITHUB (NHÁNH DEV)
# Bấm chạy ô này bất kỳ lúc nào để tự động cập nhật run_public_test.py, src/, prebuild_cache.py, ...
import os

REPO_URL = "https://github.com/JimmyK300/zalo-vecna.git"
BRANCH = "dev"

print(f"[*] Đang đồng bộ mã nguồn từ GitHub (nhánh {BRANCH})...")
if not os.path.exists(".git"):
    !git init
    !git config core.filemode false
    !git remote add origin {REPO_URL}
else:
    !git remote set-url origin {REPO_URL}

!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git reset --hard origin/{BRANCH}

print("\n✅ Code đã cập nhật thành công! Commit hiện tại:")
!git log -1 --oneline

---  
### 3. Tăng Tốc DuckDB — Tiền Xử Lý Cache (Chạy 1 lần duy nhất)
Chuyển đổi dữ liệu 13 doanh nghiệp (`dev_001` → `dev_013`) sang database `.duckdb` nén.  
👉 Giúp DuckDB **nạp dữ liệu tức thì trong 0.01 giây** thay vì phải đọc và phân tích file Excel qua Google Drive mỗi lần suy luận!

In [ ]:
# ⚡ 3.1 Tiền xử lý DuckDB Cache cho toàn bộ 13 doanh nghiệp (mất ~1 phút, chỉ cần chạy 1 lần)
!python prebuild_cache.py --data_dir public_test

---  
### 4. Chạy Thử Nghiệm Sanity Check (16 câu đầu)
Kiểm tra mô hình Qwen, kết nối DuckDB, Auto-UNION views và bộ sinh mã Python hoạt động chuẩn xác trước khi chạy toàn bộ 1,000 câu.

In [ ]:
# 4.1 Chạy thử 16 câu đầu (tự động xóa sample cũ để kiểm tra code mới nhất)
!rm -f submission_sample.jsonl
!rm -rf result/logs

# Chạy tốc độ cao với FP16 và Batch size 16 trên A100 GPU:
!python run_public_test.py \
    --data_dir public_test \
    --output submission_sample.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --fp16 \
    --batch_size 16 \
    --limit 16 \
    --overwrite

# In kết quả sinh ra của 16 câu
!cat submission_sample.jsonl

In [ ]:
# 🩺 4.2 Chẩn đoán & Xem chi tiết log suy luận (Code SQL/Python sinh ra)
import glob, json, os

log_files = sorted(glob.glob("result/logs/*.json"))
print(f"[+] Đã tìm thấy {len(log_files)} file log trong result/logs/")

# 1. In nhanh nếu có câu nào bị abstain
abstain_count = 0
for f in log_files:
    d = json.load(open(f, encoding="utf-8"))
    if d.get("final_submission", {}).get("type") == "abstain":
        abstain_count += 1
        print(f"\n⚠️ [ABSTAIN] Câu: {d['id']} | {d.get('question')}")
        print("CODE:", (d.get("generated_code") or "")[:600])
        print("LỖI:", d.get("exec_error") or d.get("error"))

if abstain_count == 0 and log_files:
    print("🎉 TUYỆT VỜI! 100% câu hỏi đều trả về kết quả hợp lệ, không có câu nào bị abstain!")

# 2. Xem chi tiết mẫu câu đầu tiên để kiểm tra code sinh ra
if log_files:
    sample_log = json.load(open(log_files[0], encoding="utf-8"))
    print(f"\n--- Mẫu Log chi tiết ({os.path.basename(log_files[0])}) ---")
    print(json.dumps(sample_log, indent=2, ensure_ascii=False))

---  
### 5. Chạy Toàn Bộ 1,000 Câu Hỏi (Full Public Test)
Lệnh này sẽ duyệt toàn bộ 1,000 câu hỏi và lưu liên tục vào file `submission.jsonl`.  
Hệ thống đã tích hợp tính năng **Tự Động Tiếp Tục (Auto-Resume)**: nếu mạng bị ngắt hoặc phiên Colab dừng giữa chừng, bạn chỉ cần bấm chạy lại ô này, script sẽ tự nhận diện các câu đã hoàn thành và chạy tiếp mà không cần làm lại từ đầu.

In [ ]:
# 🚀 5.1 CHẠY TỐC ĐỘ CAO TRÊN A100 GPU (Khuyến nghị cho Colab A100: ~15-20 phút cho 1,000 câu)
!python run_public_test.py \
    --data_dir public_test \
    --output submission.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --fp16 \
    --batch_size 16

# 💡 Lưu ý: Nếu dùng GPU T4 Colab Free, đổi sang cấu hình 4-bit bên dưới:
# !python run_public_test.py --data_dir public_test --output submission.jsonl --model_id Qwen/Qwen2.5-Coder-7B-Instruct --load_in_4bit --batch_size 4

---  
### 6. Kiểm Tra Thống Kê & Tải Kết Quả Về Máy
Kiểm tra chất lượng và đủ 1,000 dòng trước khi nộp lên cổng Zalo AI Challenge.

In [ ]:
# 6.1 Kiểm tra số lượng dòng và phân bố kiểu dữ liệu đầu ra
import os, json
from collections import Counter

if os.path.exists("submission.jsonl"):
    lines = []
    with open("submission.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                lines.append(json.loads(line))

    print(f"📊 Tổng số câu đã hoàn thành: {len(lines)} / 1000")
    types = Counter([x.get("type") for x in lines])
    print("\nPhân bố các loại đáp án:")
    for t, c in types.most_common():
        pct = (c / len(lines)) * 100
        print(f"  • {t:10}: {c:4} câu ({pct:5.1f}%)")
else:
    print("Chưa tìm thấy file submission.jsonl. Vui lòng chạy ô 5.1 trước!")

In [ ]:
# 📥 6.2 Tải file submission.jsonl về máy tính để nộp lên cổng thi Zalo
from google.colab import files

if os.path.exists("submission.jsonl"):
    files.download("submission.jsonl")
    print("✅ Đang tải file submission.jsonl về máy...")
else:
    print("Chưa có file submission.jsonl để tải về.")